## Data Cleaning

In [58]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import OneHotEncoder
import datetime
import h3
import geopandas
import json

In [38]:
df = pd.read_csv("../csv/Taxi_Trips_(2024-).csv")

C:\Users\angel\AppData\Local\Temp\ipykernel_28444\3810927188.py:1: DtypeWarning: Columns (0: Trip Miles) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../csv/Taxi_Trips_(2024-).csv")


### 1.1 Converting data

Converting numerical data to numerical data type

In [39]:
df["Trip Miles"] = df["Trip Miles"].str.replace(".", "", regex=False)
df["Trip Miles"] = df["Trip Miles"].str.replace(",", ".", regex=False)
df["Trip Miles"] = pd.to_numeric(df["Trip Miles"])

In [40]:
df["Extras"] = df["Extras"].str.replace(".", "", regex=False)
df["Extras"] = df["Extras"].str.replace(",", ".", regex=False)
df["Extras"] = df["Extras"].str.replace("$", "", regex=False)
df["Extras"] = pd.to_numeric(df["Extras"])
df["Tips"] = df["Tips"].str.replace(".", "", regex=False)
df["Tips"] = df["Tips"].str.replace(",", ".", regex=False)
df["Tips"] = df["Tips"].str.replace("$", "", regex=False)
df["Tips"] = pd.to_numeric(df["Tips"])
df["Tolls"] = df["Tolls"].str.replace(".", "", regex=False)
df["Tolls"] = df["Tolls"].str.replace(",", ".", regex=False)
df["Tolls"] = df["Tolls"].str.replace("$", "", regex=False)
df["Tolls"] = pd.to_numeric(df["Tolls"])
df["Fare"] = df["Fare"].str.replace(".", "", regex=False)
df["Fare"] = df["Fare"].str.replace(",", ".", regex=False)
df["Fare"] = df["Fare"].str.replace("$", "", regex=False)
df["Fare"] = pd.to_numeric(df["Fare"])

In [41]:
df["Trip Total"] = df["Trip Total"].str.replace(".", "", regex=False)
df["Trip Total"] = df["Trip Total"].str.replace(",", ".", regex=False)
df["Trip Total"] = df["Trip Total"].str.replace("$", "", regex=False)
df["Trip Total"] = pd.to_numeric(df["Trip Total"])

In [42]:
if isinstance(df["Trip Seconds"].dtype, pd.StringDtype) :
    df["Trip Seconds"] = df["Trip Seconds"].str.replace(".", "", regex=False)
    df["Trip Seconds"] = df["Trip Seconds"].str.replace(",", ".", regex=False)
    df["Trip Seconds"] = pd.to_numeric(df["Trip Seconds"])

Change Longitude and Latitude data to numerical data type

In [43]:
if isinstance(df["Pickup Centroid Latitude"].dtype, pd.StringDtype) :
    df["Pickup Centroid Latitude"] = df["Pickup Centroid Latitude"].str.replace(",", ".")
    df["Pickup Centroid Latitude"] = pd.to_numeric(df["Pickup Centroid Latitude"])

    df["Pickup Centroid Longitude"] = df["Pickup Centroid Longitude"].str.replace(",", ".")
    df["Pickup Centroid Longitude"] = pd.to_numeric(df["Pickup Centroid Longitude"])

    df["Dropoff Centroid Latitude"] = df["Dropoff Centroid Latitude"].str.replace(",", ".")
    df["Dropoff Centroid Latitude"] = pd.to_numeric(df["Dropoff Centroid Latitude"])

    df["Dropoff Centroid Longitude"] = df["Dropoff Centroid Longitude"].str.replace(",", ".")
    df["Dropoff Centroid Longitude"] = pd.to_numeric(df["Dropoff Centroid Longitude"])

Convert date data to datetime

In [44]:
df["Trip Start Timestamp"] = pd.to_datetime(df["Trip Start Timestamp"])
df["Trip End Timestamp"] = pd.to_datetime(df["Trip End Timestamp"])

C:\Users\angel\AppData\Local\Temp\ipykernel_28444\2590110223.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["Trip Start Timestamp"] = pd.to_datetime(df["Trip Start Timestamp"])
C:\Users\angel\AppData\Local\Temp\ipykernel_28444\2590110223.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["Trip End Timestamp"] = pd.to_datetime(df["Trip End Timestamp"])


### 1.2 Cleaning data 

Delete all trips with time seconds < 60

In [45]:
n_before = len(df)
df = df[df["Trip Seconds"] >= 60]
print(f"Dropped {n_before - len(df):,} trips below 60 seconds.")

Dropped 7,874,117 trips below 60 seconds.


Delete all trips with 0 total pay

In [46]:
n_before = len(df)
df = df[df["Trip Total"] != 0]
print(f"Dropped {n_before - len(df):,} trips with no income.")

Dropped 13,726 trips with no income.


Check if df has a location

In [47]:
df = df[df["Pickup Census Tract"].notnull() | df["Pickup Community Area"].notnull() | df["Pickup Centroid Location"].notnull()]

If Payment Type = Null change to unknown

In [48]:
print(df["Payment Type"].unique())
df["Payment Type"] = df["Payment Type"].fillna("Unknown")

<StringArray>
[       'Cash', 'Credit Card',      'Mobile',     'Unknown',      'Prcard',
   'No Charge',     'Dispute',     'Prepaid']
Length: 8, dtype: str


Drop duplicates

In [49]:
n_before = len(df)
# drop duplicates in Trip ID
df = df.drop_duplicates(subset="Trip ID")
print(f"Dropped {n_before - len(df):,} duplicate rows.")

Dropped 0 duplicate rows.


Drop implausible trips

In [50]:
n_before = len(df)
df = df[df["Trip End Timestamp"] > df["Trip Start Timestamp"]]
print(f"Dropped {n_before - len(df):,} nonsensical data.")

Dropped 2,875,319 nonsensical data.


Check if Trip Seconds makes sense for distance
(discuss how to handle nonsense data)

In [51]:
n_before = len(df)
df = df[df["Trip Miles"] / (df["Trip Seconds"]/3600) < 58]
print(f"Dropped {n_before - len(df):,} nonsensical data.")

Dropped 4,500,116 nonsensical data.


Check for missing values

In [61]:
missing = (df.isnull().sum()
           .rename("n_missing")
           .to_frame()
           .assign(pct_missing=lambda x: 100 * x["n_missing"] / len(df))
           .sort_values("pct_missing", ascending=False))

display(missing.style.background_gradient(cmap="OrRd", subset=["pct_missing"]))

,n_missing,pct_missing
Dropoff Census Tract,6604,52.180784
Pickup Census Tract,6574,51.943742
Dropoff Community Area,410,3.239570
Dropoff Centroid Longitude,317,2.504741
Dropoff Centroid Location,317,2.504741
Dropoff Centroid Latitude,317,2.504741
Extras,9,0.071113
Tolls,9,0.071113
Trip Total,9,0.071113
Fare,9,0.071113


In [63]:
df.head()

,Trip ID,Taxi ID,Trip Start Timestamp,Trip End Timestamp,Trip Seconds,Trip Miles,Pickup Census Tract,Dropoff Census Tract,Pickup Community Area,Dropoff Community Area,...,Extras,Trip Total,Payment Type,Company,Pickup Centroid Latitude,Pickup Centroid Longitude,Pickup Centroid Location,Dropoff Centroid Latitude,Dropoff Centroid Longitude,Dropoff Centroid Location
884845,4900557c918994c064f16350a91305524e8218c8,610262eb4d6e6e553bacee0777aaf61946e8e16f2e2c35...,2026-03-17 08:00:00,2026-03-17 08:15:00,900.0,6.0,NaN,NaN,6.0,32.0,...,0.0,2200.0,Credit Card,Transit Administrative Center Inc,41.944227,-87.655998,POINT (-87.6559981815 41.9442266014),41.878866,-87.625192,POINT (-87.6251921424 41.8788655841)
884853,4982757d11dc1f41c2e984ed9988aedefef3de71,199fa05b63204aa1c620c161c5cebe43b0909b1ee99864...,2026-03-17 08:00:00,2026-03-17 08:15:00,600.0,0.0,1.703184e+10,1.703108e+10,32.0,8.0,...,0.0,1050.0,Cash,Taxicab Insurance Agency LLC,41.880994,-87.632746,POINT (-87.6327464887 41.8809944707),41.890922,-87.618868,POINT (-87.6188683546 41.8909220259)
884873,35f494f7f39458ee4ca4f8535e87d2af4cc29032,b5bf5d282fa4191c68fe6552ccd45134162eacba69b016...,2026-03-17 08:00:00,2026-03-17 08:15:00,761.0,0.0,1.703108e+10,1.703184e+10,8.0,8.0,...,0.0,1.0,Cash,Tac - Yellow Cab Association,41.898332,-87.620763,POINT (-87.6207628651 41.8983317935),41.904935,-87.649907,POINT (-87.6499072264 41.9049353016)
884955,72558ab12b62933e3181598fb7431b07ebc129fb,32aa143752e0ab419df8db4d44e341571c4cad6ddd1515...,2026-03-17 08:00:00,2026-03-17 08:15:00,702.0,11.0,1.703128e+10,1.703184e+10,28.0,32.0,...,0.0,1218.0,Mobile,Blue Ribbon Taxi Association,41.879255,-87.642649,POINT (-87.642648998 41.8792550844),41.880994,-87.632746,POINT (-87.6327464887 41.8809944707)
885062,52f7e89d5c828f2c4b8638ad514dad6f1e9a7530,268102b5c5c93024c9b7ea7629e1d5d4b04cea3388961a...,2026-03-17 08:00:00,2026-03-17 08:30:00,960.0,0.0,1.703108e+10,1.703128e+10,8.0,28.0,...,0.0,2000.0,Cash,Transit Administrative Center Inc,41.898332,-87.620763,POINT (-87.6207628651 41.8983317935),41.879255,-87.642649,POINT (-87.642648998 41.8792550844)


## Preparing Data

One hot encode Payment Type

In [64]:
encoder = OneHotEncoder(sparse_output=False)
one_hot_encoded = encoder.fit_transform(df[['Payment Type']])

one_hot_df = pd.DataFrame(
    one_hot_encoded,
    columns=encoder.get_feature_names_out(['Payment Type']),
    index=df.index
)
df = pd.concat([df, one_hot_df], axis=1)
df = df.drop('Payment Type', axis=1)

Converting location to hexagons

In [65]:
# there is no trip where Pickup Census Tract has data, where Pickup Centroid Location does not have
#print(df[df["Pickup Census Tract"].notnull() & df["Pickup Centroid Location"].isnull()].count())
#print(df[df["Pickup Census Tract"].notnull() & df["Pickup Centroid Latitude"].isnull()].count())
#print(df[df["Pickup Census Tract"].notnull() & df["Pickup Centroid Longitude"].isnull()].count())
print(df[df["Pickup Community Area"].notnull() & df["Pickup Centroid Longitude"].isnull()].count())
print(df[df["Dropoff Community Area"].notnull() & df["Dropoff Centroid Longitude"].isnull()].count())

Trip ID                       0
Taxi ID                       0
Trip Start Timestamp          0
Trip End Timestamp            0
Trip Seconds                  0
Trip Miles                    0
Pickup Census Tract           0
Dropoff Census Tract          0
Pickup Community Area         0
Dropoff Community Area        0
Fare                          0
Tips                          0
Tolls                         0
Extras                        0
Trip Total                    0
Company                       0
Pickup Centroid Latitude      0
Pickup Centroid Longitude     0
Pickup Centroid Location      0
Dropoff Centroid Latitude     0
Dropoff Centroid Longitude    0
Dropoff Centroid  Location    0
Payment Type_Cash             0
Payment Type_Credit Card      0
Payment Type_Dispute          0
Payment Type_Mobile           0
Payment Type_No Charge        0
Payment Type_Prcard           0
Payment Type_Unknown          0
dtype: int64
Trip ID                       0
Taxi ID                    

City boundry block for chicago?

In [66]:
#city_bounding_box = geopandas.read_file('')
#city_bounding_box_json_string = city_bounding_box.to_json()
#city_bounding_box_json = json.loads(city_bounding_box_json_string)
#city_bounding_box_poly = city_bounding_box_json["features"][0]

In [67]:
resolution = 7

In [68]:

df["hex_loc_pick_up"] = df.apply(
    lambda row: h3.latlng_to_cell(
        row["Pickup Centroid Latitude"],
        row["Pickup Centroid Longitude"],
        resolution
    )
    if pd.notna(row["Pickup Centroid Latitude"]) and pd.notna(row["Pickup Centroid Longitude"])
    else 0,
    axis=1
)

df["hex_loc_drop_off"] = df.apply(
    lambda row: h3.latlng_to_cell(
        row["Dropoff Centroid Latitude"],
        row["Dropoff Centroid Longitude"],
        resolution
    )
    if pd.notna(row["Dropoff Centroid Latitude"]) and pd.notna(row["Dropoff Centroid Longitude"])
    else 0,
    axis=1
)